In [1]:
import sys;
print(sys.executable)

/Users/siva/anaconda3/envs/langchainai/bin/python


In [6]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# 🔐 Load API keys
load_dotenv(".env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# NOTE: gpt-3.5-turbo (used in the original notebook) has been retired.
# Swap in whichever current chat model you have access to.
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# --- Tools -------------------------------------------------------------

@tool
def simple_qa(question: str) -> str:
    """Answer a question clearly and directly, without hallucinating."""
    response = model.invoke(f"Answer clearly: {question}")
    return response.content

# Web search tool (Tavily) — replaces the deprecated TavilySearchResults
web_search = TavilySearch(max_results=3)

tools = [simple_qa, web_search]

In [7]:
#)1.Basic agent (replaces ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a helpful assistant. Use tools when they help you answer accurately.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]}
)

print(result["messages"][-1].content)

LangChain is a framework designed to help developers build applications using large language models (LLMs) by providing tools to manage prompts, chain model calls, integrate external data, and handle memory. It was created by Harrison Chase.


In [8]:
# 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "agent-types-demo-1"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    thread_config,
)
print(result["messages"][-1].content)

LangChain is a framework designed to help developers build applications that use large language models (LLMs) more effectively by managing prompts, connecting to data sources, chaining model calls, and integrating with APIs. It enables the creation of complex, intelligent applications like chatbots and question-answering systems. LangChain was created by Harrison Chase.


In [9]:
## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)
follow_up = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Now plan a 3-step study path for what I just asked about."}]},
    thread_config,  # same thread_id -> agent remembers the prior turn
)
print(follow_up["messages"][-1].content)

Here is a 3-step study path to learn about LangChain and its creator:

Step 1: Understand the Basics of LangChain
- Study the core concepts of LangChain, including its purpose, key features, and how it helps build applications with large language models.
- Explore introductory tutorials or official documentation to get a foundational understanding of the framework.

Step 2: Explore LangChain's Components and Use Cases
- Dive deeper into the components of LangChain such as prompt management, data source connections, chaining calls, and API integrations.
- Review example projects or case studies demonstrating how LangChain is used in real-world applications like chatbots and question-answering systems.

Step 3: Learn About the Creator and Community
- Research Harrison Chase, the creator of LangChain, to understand his background and vision for the framework.
- Join LangChain communities, forums, or follow relevant social media channels to stay updated on developments and engage with othe

In [10]:
## 4) Structured, multi-argument tools (replaces `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION`)
@tool
def title_maker(topic: str, tone: str = "concise") -> str:
    """Generate a title given a topic and an optional tone (e.g. 'concise', 'playful', 'formal')."""
    return f"{tone.title()} Title: {topic} in Practice"

structured_agent = create_agent(
    model=model,
    tools=[title_maker],
)

result = structured_agent.invoke(
    {"messages": [{"role": "user", "content": "Make a friendly title about LangGraph tutorials."}]}
)
print(result["messages"][-1].content)

Here is a friendly title for LangGraph tutorials: "LangGraph Tutorials in Practice." If you'd like a different style or tone, feel free to let me know!


In [11]:
## 5) Native function / tool calling (replaces `OPENAI_FUNCTIONS`)
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Search the web for LangGraph docs and give me 3 bullets."}]},
    stream_mode="values",
):
    last = step["messages"][-1]
    if getattr(last, "tool_calls", None):
        print("Tool call(s):", [tc["name"] for tc in last.tool_calls])
    elif last.content:
        print("Final answer:\n", last.content)

Final answer:
 Search the web for LangGraph docs and give me 3 bullets.
Tool call(s): ['tavily_search']
Final answer:
 {"query": "LangGraph docs", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.scaler.com/topics/langgraph-doc-explained", "title": "LangGraph Docs Explained | The 20% You’ll Use 80% of the Time", "content": "Topics Covered\n\nLooking for the official LangGraph documentation? It lives at docs.langchain.com start with the overview, then the quickstart. That's the honest answer, in sentence one.  \n Now the reason you're probably here: the LangGraph docs are excellent as a reference, but nobody tells you the reading order. Dozens of guide pages, two APIs (Graph and Functional), a separate API reference site, a platform layer and yet about six core concepts power almost every real project. [...] The LangGraph documentation stops being intimidating the moment it has a map: docs.langchain.com for concepts, reference.langchain.com for